In [ ]:
import polars as pl

In [ ]:
df_batch_measurements = pl.scan_parquet("../data/batch_measurements.parquet")
df_measurements = pl.scan_parquet("../data/measurements.parquet")

print("=== Batch Measurements ===")
print(df_batch_measurements.head(2).collect())
print("=== Measurements ===")
print(df_measurements.head(2).collect())

Was there in 2022 an injured polar bear older than 15 (i.e. a senior polar bear)?


In [ ]:
(
    df_batch_measurements.filter(pl.col("vet_health_check") == "INJURED")
    .filter(pl.col("age") > 15)
    .filter(pl.col("timestamp").dt.year() == 2022)
    .select("name")
    .unique()
).collect()

In [ ]:
# Alernative: define expression

injured_senior_expr = (
    (pl.col("vet_health_check") == "INJURED")
    & (pl.col("age") > 15)
    & (pl.col("timestamp").dt.year() == 2022)
)

df_batch_measurements.filter(injured_senior_expr).select("name").unique().collect()

In [ ]:
# Alternative: Clean the data first


def clean(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(pl.col("name").str.to_titlecase())


df_batch_measurements.pipe(clean).filter(injured_senior_expr).select(
    "name"
).unique().collect()

Bonus question: can you answer the same question using Polars [categorical types](https://docs.pola.rs/user-guide/expressions/categorical-data-and-enums/#data-type-categorical)?


In [ ]:
# pl.Categorical vs pl.Enum - Both store strings as integers for efficiency
#
# pl.Categorical:
#   - Categories are determined at runtime from the data
#   - Memory efficient: stores integers instead of repeating strings
#   - Flexible: new categories can appear in different DataFrames
#   - Disadvantage: comparing Categoricals from different sources can fail
#     (they may have different internal integer mappings)
#
# pl.Enum:
#   - Categories are defined upfront
#   - Validates data: casting fails if a value isn't in the allowed set
#   - Safe comparisons: all Enums of the same type share the same mapping
#   - Disadvantage: less flexible, must know all possible values beforehand

vet_health_check_enum = pl.Enum(["SICK", "INJURED", "HEALTHY"])
life_stage_enum = pl.Enum(["CUB", "JUV", "ADULT", "SENIOR"])

(
    df_batch_measurements.pipe(clean)
    # Cast to categorical types
    .with_columns(
        pl.col("name").cast(pl.Categorical),  # Flexible - names vary
        pl.col("vet_health_check").cast(vet_health_check_enum),  # Fixed set
        pl.col("life_stage").cast(life_stage_enum),  # Fixed set
    )
    .filter(pl.col("timestamp").dt.year() == 2022)
    .filter(pl.col("vet_health_check") == "INJURED")
    .filter(pl.col("life_stage") == "SENIOR")
    .select("name")
    .unique()
    .collect()
)

How many times was Blizzard Bob's name capitalized in the batch measurements?


In [ ]:
name = "Blizzard Bob"

(
    df_batch_measurements.filter(pl.col("name").str.to_lowercase() == name.lower())
    .group_by("name")
    .agg(pl.col("name").count().alias("count"))
    .collect()
)

Was Chilly Willy ever sick with a temperature above 40 degrees? (Hint: you might need to perform some data wrangling by performing a union and downfill.)


In [ ]:
name = "Chilly Willy"

(
    # Union
    # Combine the batch measurements, containing health checks,
    # with the real-time measurements containing sensor values
    pl.concat(
        [
            df_batch_measurements.pipe(clean),
            df_measurements,
        ],
        how="diagonal",
    )
    .filter(pl.col("name") == name)
    # Downfill
    # Create a (sorted) timeline for all Chilly Willy events,
    # containig both health check and the temperature reading
    .sort("timestamp")
    .with_columns(pl.col("temperature", "vet_health_check").forward_fill())
    # and Filter
    .filter((pl.col("vet_health_check") == "SICK") & (pl.col("temperature") > 40))
    .select("name", "temperature", "vet_health_check")
    # .head(5)
    .collect()
)

In [ ]:
# Alternative: using join_asof
#
# join_asof is an "asof join" - it matches rows based on approximate key values
# rather than requiring exact matches. Useful for time-series data where
# events don't happen at exactly the same time.
#
# How it works:
#   1. Both DataFrames must be sorted by the join column (timestamp here)
#   2. For each row in the LEFT DataFrame (df_measurements), it finds the
#      matching row in the RIGHT DataFrame (df_batch_measurements) with:
#      - Same "name" (the `by` parameter - exact match required)
#      - Closest "timestamp" (the `on` parameter - approximate match)
#
# Strategies:
#   - "backward": closest timestamp <= measurement time (only past)
#   - "forward":  closest timestamp >= measurement time (only future)
#   - "nearest":  closest timestamp in either direction
#
# Using strategy="backward" is equivalent to the concat + forward_fill approach:
# both assign each measurement the most recent health check that occurred
# before (or at) the measurement time.

(
    df_measurements.sort("timestamp")
    .join_asof(
        df_batch_measurements.pipe(clean).sort("timestamp"),
        by="name",  # Exact match on name
        on="timestamp",  # Approximate match on timestamp
        strategy="backward",  # Most recent health check before this measurement
    )
    .select("name", "temperature", "vet_health_check")
    .filter(pl.col("temperature") > 40, name="Chilly Willy", vet_health_check="SICK")
    .collect()
)